# Problem 2 — Transformer from scratch
No `nn.MultiheadAttention`, `nn.Transformer` or `nn.TransformerEncoder` is used. Run from the project root.

In [1]:
import os, sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
from config import *
from problem2.core import *
import pandas as pd
print("Device:",get_device(),"d:",MODEL_WIDTH,"heads:",NUM_HEADS,"layers:",NUM_LAYERS)

Device: cpu d: 64 heads: 4 layers: 4


## Task 1 — Scaled dot-product attention and hand-checkable 3-token example

In [2]:
print(test_attention())

Q, K, V, scores, weights, out = three_token_example()

print("Q:", Q, "\nK:", K, "\nV:", V)
print("Scores:", scores)
print("Weights:", weights)
print("Output:", out)

{'max_error_vs_torch': 1.7881393432617188e-07, 'row_sum_max_error': 1.1920928955078125e-07, 'future_weight_max': 0.0}
Q: tensor([[1., 0.],
        [0., 1.],
        [1., 1.]]) 
K: tensor([[1., 0.],
        [0., 1.],
        [1., 1.]]) 
V: tensor([[1., 0.],
        [0., 2.],
        [3., 1.]])
Scores: tensor([[0.7071, 0.0000, 0.7071],
        [0.0000, 0.7071, 0.7071],
        [0.7071, 0.7071, 1.4142]])
Weights: tensor([[0.4011, 0.1978, 0.4011],
        [0.1978, 0.4011, 0.4011],
        [0.2483, 0.2483, 0.5035]])
Output: tensor([[1.6044, 0.7967],
        [1.4011, 1.2033],
        [1.7587, 1.0000]])


## Task 2 — Multi-head attention and pre-LayerNorm encoder block

In [3]:
block=EncoderBlock(MODEL_WIDTH,NUM_HEADS,FF_WIDTH)
x=torch.randn(2,12,MODEL_WIDTH)
y=block(x)
assert y.shape==x.shape
print("Encoder output:",y.shape,"parameters:",sum(p.numel() for p in block.parameters()))
print("Linear weights-only formula:",4*MODEL_WIDTH**2+2*MODEL_WIDTH*FF_WIDTH)

Encoder output: torch.Size([2, 12, 64]) parameters: 49984
Linear weights-only formula: 49152


## Task A — HAR encoder classifier and positional encoding ablation

In [4]:
har_transformer_results=train_har_transformer()
display(har_transformer_results)
print("Transformer HAR mean ± sample std by positional encoding:")
display(har_transformer_results.groupby("model")[["test_acc","macro_f1","params","sec_per_epoch"]].agg(["mean","std"]))

,problem,model,seed,seq_len,test_acc,macro_f1,test_mse,params,sec_per_epoch,val_loss_nats,bits_per_char,notes
0,P2-HAR,Transformer-none,42,128,0.887682,0.886330,,201158,53.751000,,,epochs=10; best validation accuracy=0.8786
1,P2-HAR,Transformer-sinusoidal,42,128,0.882932,0.881136,,201158,49.822736,,,epochs=10; best validation accuracy=0.8836
2,P2-HAR,Transformer-learned,42,128,0.907363,0.907042,,209478,48.622226,,,epochs=10; best validation accuracy=0.8914
3,P2-HAR,Transformer-none,123,128,0.893112,0.892511,,201158,48.190055,,,epochs=10; best validation accuracy=0.9492
4,P2-HAR,Transformer-sinusoidal,123,128,0.889040,0.887390,,201158,48.739425,,,epochs=10; best validation accuracy=0.9547
5,P2-HAR,Transformer-learned,123,128,0.876145,0.875404,,209478,48.966412,,,epochs=10; best validation accuracy=0.9581
6,P2-HAR,Transformer-none,2026,128,0.900238,0.900064,,201158,51.701531,,,epochs=10; best validation accuracy=0.9615
7,P2-HAR,Transformer-sinusoidal,2026,128,0.891415,0.890988,,201158,50.844869,,,epochs=10; best validation accuracy=0.9750
8,P2-HAR,Transformer-learned,2026,128,0.886664,0.885183,,209478,48.864266,,,epochs=10; best validation accuracy=0.9708


Transformer HAR mean ± sample std by positional encoding:


test_acc            macro_f1              params       \
                            mean       std      mean       std      mean  std   
model                                                                           
Transformer-learned     0.890058  0.015883  0.889210  0.016199  209478.0  0.0   
Transformer-none        0.893677  0.006297  0.892969  0.006878  201158.0  0.0   
Transformer-sinusoidal  0.887795  0.004376  0.886505  0.004985  201158.0  0.0   

                       sec_per_epoch            
                                mean       std  
model                                           
Transformer-learned        48.817635  0.176768  
Transformer-none           51.214195  2.812321  
Transformer-sinusoidal     49.802343  1.052870

## Task B — Tiny Shakespeare character model

In [5]:
model,lm_metrics,samples=train_lm()
print(lm_metrics)
print("Generated samples:",len(samples),"each 500 characters (excluding prompt)")

{'val_loss_nats': 2.0740384221076966, 'bits_per_char': 2.9922049461879445, 'seconds': 1032.746976852417, 'params': 216641}
Generated samples: 3 each 500 characters (excluding prompt)


## Task 5 — Attention maps, head ablation, causal-mask ablation, context scaling

train,val,stoi,itos=load_shakespeare()
plot_attention_heatmaps(model,stoi,itos)
display(run_head_ablation())
display(run_no_mask_ablation())
display(run_context_benchmark())